<a href="https://colab.research.google.com/github/amathie5/PPS-Project-/blob/main/pps_project_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Parameters
horizon = 12    #in months
w_i = 90        # Initial number of workers
p = 10          #Production rate per worker per month
inv_i = 900     # Initial inventory
inv_t = 1000    # Final inventory target by end of December
hold = 1000     # Inventory holding cost per watch per month (security/insurance)
wage_r = 7000   # Regular wage/worker/month
hiring = 50000  # Hiring cost per worker
layoff = 25000  # layoff cost per worker
max_overtime = 0.2  # overtime allowance......
overtime_cost = 14000 # 2x wage_r
subcontracting_cost = 15000 # CHF per watch
max_contracting = 300 # watches per month
overtime_months = [3,5,9,12] # march, may, sept, dec
subcontracting_months = [6,7,10,12] # june, july, oct, dec
demand = [900,950,1200,1050,1100,1300,1250,1100,1300,1450,1500,1700]

In [ ]:
pip install pulp numpy pandas matplotlib


In [ ]:
import numpy as np
import pandas as pd

def simulate_plan(strategy):

    months = horizon
    workers = np.zeros(months)
    hired = np.zeros(months)
    fired = np.zeros(months)
    production = np.zeros(months)
    overtime_prod = np.zeros(months) # Nouveau tableau pour l'overtime
    inventory = np.zeros(months+1)

    inventory[0] = inv_i

    # ----------------------------
    # LEVEL STRATEGY
    # ----------------------------
    if strategy == "level":

        production_per_month = w_i * p
        workers[:] = w_i

        for t in range(months):

            production[t] = production_per_month
            inventory[t+1] = inventory[t] + production[t] - demand[t]

    # ----------------------------
    # CHASE STRATEGY
    # ----------------------------
    elif strategy == "chase":

        for t in range(months):

            workers_needed = np.ceil(demand[t] / p)
            workers[t] = workers_needed

            if t == 0:
                hired[t] = max(0, workers[t] - w_i)
                fired[t] = max(0, w_i - workers[t])
            else:
                hired[t] = max(0, workers[t] - workers[t-1])
                fired[t] = max(0, workers[t-1] - workers[t])

            production[t] = workers[t] * p
            inventory[t+1] = inventory[t] + production[t] - demand[t]

    # ----------------------------
    # MIXED STRATEGY (Level + Overtime)
    # ----------------------------
    elif strategy == "mixed":

        # workers constants = demande moyenne
        avg_demand = sum(demand) / months
        level_workers = np.ceil(avg_demand / p)

        workers[:] = level_workers

        hired[0] = max(0, level_workers - w_i)
        fired[0] = max(0, w_i - level_workers)

        for t in range(months):

            reg_prod = workers[t] * p
            production[t] = reg_prod

            shortage = demand[t] - (inventory[t] + reg_prod)

            if shortage > 0 and (t + 1) in overtime_months:
                max_ot = reg_prod * max_overtime
                ot_used = min(shortage, max_ot)

                overtime_prod[t] = ot_used
                production[t] += ot_used

            inventory[t+1] = inventory[t] + production[t] - demand[t]

    # ----------------------------
    # FINAL INVENTORY CONSTRAINT
    # ----------------------------

    adjustment = inv_t - inventory[-1]
    production[-1] += adjustment
    inventory[-1] = inv_t

    # ----------------------------
    # COSTS
    # ----------------------------

    wage_cost = workers * wage_r
    hiring_cost = hired * hiring
    firing_cost = fired * layoff

    ot_unit_cost = overtime_cost / p
    ot_cost = overtime_prod * ot_unit_cost

    inventory_cost = inventory[1:] * hold

    total_cost = wage_cost + hiring_cost + firing_cost + inventory_cost + ot_cost

    results = pd.DataFrame({
        "Month": range(1, months+1),
        "Workers": workers,
        "Hired": hired,
        "Fired": fired,
        "Regular Prod": production - overtime_prod,
        "Overtime Prod": overtime_prod,
        "Total Prod": production,
        "Demand": demand,
        "Inventory": inventory[1:],
        "Total Cost": total_cost
    })

    summary = {
        "Total Wage": wage_cost.sum(),
        "Total Hiring": hiring_cost.sum(),
        "Total Layoff": firing_cost.sum(),
        "Total Overtime": ot_cost.sum(),
        "Total Inventory": inventory_cost.sum(),
        "Grand Total": total_cost.sum()
    }

    return results, summary

In [ ]:
#level strategy
level_table, level_cost = simulate_plan("level")

display(level_table)
display(level_cost)

,Month,Workers,Hired,Fired,Regular Prod,Overtime Prod,Total Prod,Demand,Inventory,Total Cost
0,1,90.0,0.0,0.0,900.0,0.0,900.0,900,900.0,1530000.0
1,2,90.0,0.0,0.0,900.0,0.0,900.0,950,850.0,1480000.0
2,3,90.0,0.0,0.0,900.0,0.0,900.0,1200,550.0,1180000.0
3,4,90.0,0.0,0.0,900.0,0.0,900.0,1050,400.0,1030000.0
4,5,90.0,0.0,0.0,900.0,0.0,900.0,1100,200.0,830000.0
5,6,90.0,0.0,0.0,900.0,0.0,900.0,1300,-200.0,430000.0
6,7,90.0,0.0,0.0,900.0,0.0,900.0,1250,-550.0,80000.0
7,8,90.0,0.0,0.0,900.0,0.0,900.0,1100,-750.0,-120000.0
8,9,90.0,0.0,0.0,900.0,0.0,900.0,1300,-1150.0,-520000.0
9,10,90.0,0.0,0.0,900.0,0.0,900.0,1450,-1700.0,-1070000.0


{'Total Wage': np.float64(7560000.0),
 'Total Hiring': np.float64(0.0),
 'Total Layoff': np.float64(0.0),
 'Total Overtime': np.float64(0.0),
 'Total Inventory': np.float64(-2750000.0),
 'Grand Total': np.float64(4810000.0)}

In [ ]:
chase_table, chase_cost = simulate_plan("chase")

display(chase_table)
display(chase_cost)

,Month,Workers,Hired,Fired,Regular Prod,Overtime Prod,Total Prod,Demand,Inventory,Total Cost
0,1,90.0,0.0,0.0,900.0,0.0,900.0,900,900.0,1530000.0
1,2,95.0,5.0,0.0,950.0,0.0,950.0,950,900.0,1815000.0
2,3,120.0,25.0,0.0,1200.0,0.0,1200.0,1200,900.0,2990000.0
3,4,105.0,0.0,15.0,1050.0,0.0,1050.0,1050,900.0,2010000.0
4,5,110.0,5.0,0.0,1100.0,0.0,1100.0,1100,900.0,1920000.0
5,6,130.0,20.0,0.0,1300.0,0.0,1300.0,1300,900.0,2810000.0
6,7,125.0,0.0,5.0,1250.0,0.0,1250.0,1250,900.0,1900000.0
7,8,110.0,0.0,15.0,1100.0,0.0,1100.0,1100,900.0,2045000.0
8,9,130.0,20.0,0.0,1300.0,0.0,1300.0,1300,900.0,2810000.0
9,10,145.0,15.0,0.0,1450.0,0.0,1450.0,1450,900.0,2665000.0


{'Total Wage': np.float64(10360000.0),
 'Total Hiring': np.float64(5750000.0),
 'Total Layoff': np.float64(875000.0),
 'Total Overtime': np.float64(0.0),
 'Total Inventory': np.float64(10900000.0),
 'Grand Total': np.float64(27885000.0)}

In [ ]:
mixed_table, mixed_cost = simulate_plan("mixed")
display(mixed_table)
display(mixed_cost)

,Month,Workers,Hired,Fired,Regular Prod,Overtime Prod,Total Prod,Demand,Inventory,Total Cost
0,1,124.0,34.0,0.0,1240.0,0.0,1240.0,900,1240.0,3808000.0
1,2,124.0,0.0,0.0,1240.0,0.0,1240.0,950,1530.0,2398000.0
2,3,124.0,0.0,0.0,1240.0,0.0,1240.0,1200,1570.0,2438000.0
3,4,124.0,0.0,0.0,1240.0,0.0,1240.0,1050,1760.0,2628000.0
4,5,124.0,0.0,0.0,1240.0,0.0,1240.0,1100,1900.0,2768000.0
5,6,124.0,0.0,0.0,1240.0,0.0,1240.0,1300,1840.0,2708000.0
6,7,124.0,0.0,0.0,1240.0,0.0,1240.0,1250,1830.0,2698000.0
7,8,124.0,0.0,0.0,1240.0,0.0,1240.0,1100,1970.0,2838000.0
8,9,124.0,0.0,0.0,1240.0,0.0,1240.0,1300,1910.0,2778000.0
9,10,124.0,0.0,0.0,1240.0,0.0,1240.0,1450,1700.0,2568000.0


{'Total Wage': np.float64(10416000.0),
 'Total Hiring': np.float64(1700000.0),
 'Total Layoff': np.float64(0.0),
 'Total Overtime': np.float64(0.0),
 'Total Inventory': np.float64(19690000.0),
 'Grand Total': np.float64(31806000.0)}

In [ ]:
#Level strategy

#'The level strategy keeps the orkforce constant at 90 workers and produces 900 watches per month. However, demand exceeds production in several months, resulting in significant backorders. 2By the end of the year, the inventory level becomes negative, indicating that the strategy cannot satisfy demand'#

#Chase strategy

#The chase strategy adjusts the workforce to match demand each month. This eliminates stockouts and keeps inventory constant. However, it requires frequent hiring and layoffs, leading to high workforce instability and additional labor costs.

In [ ]:
#---------------------------
# Model Base
#---------------------------


from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpInteger, LpContinuous, LpStatus, value
import pandas as pd
import numpy as np

def solve_lp_model_1(
    demand,
    p,
    inv_i,
    inv_t,
    w_i,
    wage_r,
    hiring,
    layoff,
    hold
):
    months = len(demand)

    # Initialisation du modèle
    model = LpProblem("Aurelius_Base_Model", LpMinimize)

    # Variables de décision
    W = LpVariable.dicts("W", range(months), lowBound=0, cat=LpInteger)
    H = LpVariable.dicts("H", range(months), lowBound=0, cat=LpInteger)
    L = LpVariable.dicts("L", range(months), lowBound=0, cat=LpInteger)
    P = LpVariable.dicts("P", range(months), lowBound=0, cat=LpContinuous)
    I = LpVariable.dicts("I", range(months), lowBound=0, cat=LpContinuous)

    # Fonction objectif : Minimiser le coût total
    model += lpSum(
        W[t] * wage_r +
        H[t] * hiring +
        L[t] * layoff +
        I[t] * hold
        for t in range(months)
    ), "Total_Cost"

    # Contraintes
    for t in range(months):

        # 1. Contrainte de capacité de production
        model += P[t] == W[t] * p, f"Prod_Capacity_Month_{t}"

        # 2. Équilibre de la main-d'œuvre et de l'inventaire
        if t == 0:
            model += W[t] == w_i + H[t] - L[t], "Workforce_Balance_Month_0"
            model += I[t] == inv_i + P[t] - demand[t], "Inventory_Balance_Month_0"
        else:
            model += W[t] == W[t-1] + H[t] - L[t], f"Workforce_Balance_Month_{t}"
            model += I[t] == I[t-1] + P[t] - demand[t], f"Inventory_Balance_Month_{t}"

    # 3. Cible d'inventaire à la fin de l'horizon (Décembre)
    model += I[months - 1] >= inv_t, "Final_Inventory_Target"

    # Résolution
    model.solve()

    status = LpStatus[model.status]

    # Extraction des résultats si la solution est optimale
    if status == "Optimal":
        df = pd.DataFrame({
            "Month": np.arange(1, months+1),
            "Workers": [W[t].varValue for t in range(months)],
            "Hired": [H[t].varValue for t in range(months)],
            "Laid off": [L[t].varValue for t in range(months)],
            "Produced": [P[t].varValue for t in range(months)],
            "Demand": demand,
            "End Inventory": [I[t].varValue for t in range(months)],
        })

        summary = {
            "Total Wage": sum(W[t].varValue * wage_r for t in range(months)),
            "Total Hiring": sum(H[t].varValue * hiring for t in range(months)),
            "Total Layoff": sum(L[t].varValue * layoff for t in range(months)),
            "Total Inventory": sum(I[t].varValue * hold for t in range(months)),
            "Grand Total": value(model.objective)
        }
    else:
        df = pd.DataFrame()
        summary = {}

    return df, summary, status

In [ ]:
df_model1, summary_model1, status_model1 = solve_lp_model_1(demand, p, inv_i, inv_t, w_i, wage_r, hiring, layoff, hold)
display(df_model1)
display(summary_model1)

,Month,Workers,Hired,Laid off,Produced,Demand,End Inventory
0,1,80.0,0.0,10.0,800.0,900,800.0
1,2,80.0,0.0,0.0,800.0,950,650.0
2,3,80.0,0.0,0.0,800.0,1200,250.0
3,4,80.0,0.0,0.0,800.0,1050,0.0
4,5,115.0,35.0,0.0,1150.0,1100,50.0
5,6,125.0,10.0,0.0,1250.0,1300,0.0
6,7,125.0,0.0,0.0,1250.0,1250,0.0
7,8,125.0,0.0,0.0,1250.0,1100,150.0
8,9,125.0,0.0,0.0,1250.0,1300,100.0
9,10,185.0,60.0,0.0,1850.0,1450,500.0


{'Total Wage': 10430000.0,
 'Total Hiring': 5250000.0,
 'Total Layoff': 250000.0,
 'Total Inventory': 4350000.0,
 'Grand Total': 20280000.0}

In [ ]:
#-------------------------
# Model 2 : overtime only
#-------------------------


def solve_lp_model_2(
    demand,
    p,
    inv_i,
    inv_t,
    w_i,
    wage_r,
    hiring,
    layoff,
    hold,
    overtime_cost,
    max_overtime,
    overtime_months
):
    months = len(demand)

    # Initialisation du modèle
    model = LpProblem("Aurelius_Model_2_Overtime", LpMinimize)

    # Variables de décision
    W = LpVariable.dicts("W", range(months), lowBound=0, cat=LpInteger)
    H = LpVariable.dicts("H", range(months), lowBound=0, cat=LpInteger)
    L = LpVariable.dicts("L", range(months), lowBound=0, cat=LpInteger)
    P = LpVariable.dicts("P", range(months), lowBound=0, cat=LpContinuous)
    I = LpVariable.dicts("I", range(months), lowBound=0, cat=LpContinuous)
    O = LpVariable.dicts("O", range(months), lowBound=0, cat=LpContinuous) # Variable pour l'overtime

    # Coût unitaire de l'overtime (coût mensuel / production normale)
    ot_unit_cost = overtime_cost / p

    # Fonction objectif : Minimiser le coût total (incluant l'overtime)
    model += lpSum(
        W[t] * wage_r +
        H[t] * hiring +
        L[t] * layoff +
        I[t] * hold +
        O[t] * ot_unit_cost
        for t in range(months)
    ), "Total_Cost"

    # Contraintes
    for t in range(months):

        # 1. Contrainte de capacité de production régulière
        model += P[t] == W[t] * p, f"Reg_Prod_Capacity_Month_{t}"

        # 2. Limites d'heures supplémentaires selon les mois autorisés
        if (t + 1) in overtime_months:
            # L'overtime est limité par un pourcentage de la capacité régulière des travailleurs actifs ce mois-là
            model += O[t] <= W[t] * p * max_overtime, f"Max_Overtime_Month_{t}"
        else:
            model += O[t] == 0, f"No_Overtime_Month_{t}"

        # 3. Équilibre de la main-d'œuvre et de l'inventaire (en ajoutant O[t] aux entrées de stock)
        if t == 0:
            model += W[t] == w_i + H[t] - L[t], "Workforce_Balance_Month_0"
            model += I[t] == inv_i + P[t] + O[t] - demand[t], "Inventory_Balance_Month_0"
        else:
            model += W[t] == W[t-1] + H[t] - L[t], f"Workforce_Balance_Month_{t}"
            model += I[t] == I[t-1] + P[t] + O[t] - demand[t], f"Inventory_Balance_Month_{t}"

    # 4. Cible d'inventaire à la fin de l'horizon (Décembre)
    model += I[months - 1] >= inv_t, "Final_Inventory_Target"

    # Résolution
    model.solve()

    status = LpStatus[model.status]

    # Extraction des résultats si la solution est optimale
    if status == "Optimal":
        df = pd.DataFrame({
            "Month": np.arange(1, months+1),
            "Workers": [W[t].varValue for t in range(months)],
            "Hired": [H[t].varValue for t in range(months)],
            "Laid off": [L[t].varValue for t in range(months)],
            "Reg. Prod": [P[t].varValue for t in range(months)],
            "Overtime Prod": [O[t].varValue for t in range(months)],
            "Total Prod": [P[t].varValue + O[t].varValue for t in range(months)],
            "Demand": demand,
            "End Inventory": [I[t].varValue for t in range(months)],
        })

        summary = {
            "Total Wage": sum(W[t].varValue * wage_r for t in range(months)),
            "Total Hiring": sum(H[t].varValue * hiring for t in range(months)),
            "Total Layoff": sum(L[t].varValue * layoff for t in range(months)),
            "Total Overtime": sum(O[t].varValue * ot_unit_cost for t in range(months)),
            "Total Inventory": sum(I[t].varValue * hold for t in range(months)),
            "Grand Total": value(model.objective)
        }
    else:
        df = pd.DataFrame()
        summary = {}

    return df, summary, status

In [ ]:
df_model2, summary_model2, status_model2 = solve_lp_model_2(demand, p, inv_i, inv_t, w_i, wage_r, hiring, layoff, hold, overtime_cost, max_overtime,overtime_months)
display(df_model2)
display(summary_model2)

,Month,Workers,Hired,Laid off,Reg. Prod,Overtime Prod,Total Prod,Demand,End Inventory
0,1,80.0,0.0,10.0,800.0,0.0,800.0,900,800.0
1,2,80.0,0.0,0.0,800.0,0.0,800.0,950,650.0
2,3,80.0,0.0,0.0,800.0,0.0,800.0,1200,250.0
3,4,80.0,0.0,0.0,800.0,0.0,800.0,1050,0.0
4,5,115.0,35.0,0.0,1150.0,0.0,1150.0,1100,50.0
5,6,125.0,10.0,0.0,1250.0,0.0,1250.0,1300,0.0
6,7,125.0,0.0,0.0,1250.0,0.0,1250.0,1250,0.0
7,8,125.0,0.0,0.0,1250.0,0.0,1250.0,1100,150.0
8,9,126.0,1.0,0.0,1260.0,4.0,1264.0,1300,114.0
9,10,173.0,47.0,0.0,1730.0,0.0,1730.0,1450,394.0


{'Total Wage': 10185000.0,
 'Total Hiring': 4650000.0,
 'Total Layoff': 250000.0,
 'Total Overtime': 490000.0,
 'Total Inventory': 4032000.0,
 'Grand Total': 19607000.0}

In [ ]:
#--------------------------------
# Model 3 : subcontracting only
#---------------------------------

def solve_lp_model_3(
    demand,
    p,
    inv_i,
    inv_t,
    w_i,
    wage_r,
    hiring,
    layoff,
    hold,
    subcontracting_cost,
    max_contracting,
    subcontracting_months
):
    months = len(demand)

    # Initialisation du modèle
    model = LpProblem("Aurelius_Model_3_Subcontracting", LpMinimize)

    # Variables de décision
    W = LpVariable.dicts("W", range(months), lowBound=0, cat=LpInteger)
    H = LpVariable.dicts("H", range(months), lowBound=0, cat=LpInteger)
    L = LpVariable.dicts("L", range(months), lowBound=0, cat=LpInteger)
    P = LpVariable.dicts("P", range(months), lowBound=0, cat=LpContinuous)
    I = LpVariable.dicts("I", range(months), lowBound=0, cat=LpContinuous)
    S = LpVariable.dicts("S", range(months), lowBound=0, cat=LpContinuous) # Variable pour la sous-traitance

    # Fonction objectif : Minimiser le coût total (incluant la sous-traitance)
    model += lpSum(
        W[t] * wage_r +
        H[t] * hiring +
        L[t] * layoff +
        I[t] * hold +
        S[t] * subcontracting_cost
        for t in range(months)
    ), "Total_Cost"

    # Contraintes
    for t in range(months):

        # 1. Contrainte de capacité de production régulière
        model += P[t] == W[t] * p, f"Prod_Capacity_Month_{t}"

        # 2. Limites de sous-traitance selon les mois autorisés
        if (t + 1) in subcontracting_months:
            model += S[t] <= max_contracting, f"Max_Subcontracting_Month_{t}"
        else:
            model += S[t] == 0, f"No_Subcontracting_Month_{t}"

        # 3. Équilibre de la main-d'œuvre et de l'inventaire (en ajoutant S[t] aux entrées de stock)
        if t == 0:
            model += W[t] == w_i + H[t] - L[t], "Workforce_Balance_Month_0"
            model += I[t] == inv_i + P[t] + S[t] - demand[t], "Inventory_Balance_Month_0"
        else:
            model += W[t] == W[t-1] + H[t] - L[t], f"Workforce_Balance_Month_{t}"
            model += I[t] == I[t-1] + P[t] + S[t] - demand[t], f"Inventory_Balance_Month_{t}"

    # 4. Cible d'inventaire à la fin de l'horizon (Décembre)
    model += I[months - 1] >= inv_t, "Final_Inventory_Target"

    # Résolution
    model.solve()

    status = LpStatus[model.status]

    # Extraction des résultats si la solution est optimale
    if status == "Optimal":
        df = pd.DataFrame({
            "Month": np.arange(1, months+1),
            "Workers": [W[t].varValue for t in range(months)],
            "Hired": [H[t].varValue for t in range(months)],
            "Laid off": [L[t].varValue for t in range(months)],
            "Reg. Prod": [P[t].varValue for t in range(months)],
            "Subcontracted": [S[t].varValue for t in range(months)],
            "Total Prod": [P[t].varValue + S[t].varValue for t in range(months)],
            "Demand": demand,
            "End Inventory": [I[t].varValue for t in range(months)],
        })

        summary = {
            "Total Wage": sum(W[t].varValue * wage_r for t in range(months)),
            "Total Hiring": sum(H[t].varValue * hiring for t in range(months)),
            "Total Layoff": sum(L[t].varValue * layoff for t in range(months)),
            "Total Subcontracting": sum(S[t].varValue * subcontracting_cost for t in range(months)),
            "Total Inventory": sum(I[t].varValue * hold for t in range(months)),
            "Grand Total": value(model.objective)
        }
    else:
        df = pd.DataFrame()
        summary = {}

    return df, summary, status

In [ ]:
df_model3, summary_model3, status_model3 = solve_lp_model_3(demand, p, inv_i, inv_t, w_i, wage_r, hiring, layoff, hold, subcontracting_cost, max_contracting, subcontracting_months)
display(df_model3)
display(summary_model3)


,Month,Workers,Hired,Laid off,Reg. Prod,Subcontracted,Total Prod,Demand,End Inventory
0,1,80.0,0.0,10.0,800.0,0.0,800.0,900,800.0
1,2,80.0,0.0,0.0,800.0,0.0,800.0,950,650.0
2,3,80.0,0.0,0.0,800.0,0.0,800.0,1200,250.0
3,4,80.0,0.0,0.0,800.0,0.0,800.0,1050,0.0
4,5,115.0,35.0,0.0,1150.0,0.0,1150.0,1100,50.0
5,6,125.0,10.0,0.0,1250.0,0.0,1250.0,1300,0.0
6,7,125.0,0.0,0.0,1250.0,0.0,1250.0,1250,0.0
7,8,125.0,0.0,0.0,1250.0,0.0,1250.0,1100,150.0
8,9,125.0,0.0,0.0,1250.0,0.0,1250.0,1300,100.0
9,10,185.0,60.0,0.0,1850.0,0.0,1850.0,1450,500.0


{'Total Wage': 10430000.0,
 'Total Hiring': 5250000.0,
 'Total Layoff': 250000.0,
 'Total Subcontracting': 0.0,
 'Total Inventory': 4350000.0,
 'Grand Total': 20280000.0}

In [ ]:
#-----------------------------------
# Model 4
#----------------------------------
def solve_lp_model_4(
    demand,
    p,
    inv_i,
    inv_t,
    w_i,
    wage_r,
    hiring,
    layoff,
    hold,
    overtime_cost,
    max_overtime,
    overtime_months,
    subcontracting_cost,
    max_contracting,
    subcontracting_months
):
    months = len(demand)

    # Initialisation du modèle
    model = LpProblem("Aurelius_Model_4_Full_Flexibility", LpMinimize)

    # Variables de décision
    W = LpVariable.dicts("W", range(months), lowBound=0, cat=LpInteger)
    H = LpVariable.dicts("H", range(months), lowBound=0, cat=LpInteger)
    L = LpVariable.dicts("L", range(months), lowBound=0, cat=LpInteger)
    P = LpVariable.dicts("P", range(months), lowBound=0, cat=LpContinuous)
    I = LpVariable.dicts("I", range(months), lowBound=0, cat=LpContinuous)
    O = LpVariable.dicts("O", range(months), lowBound=0, cat=LpContinuous) # Overtime
    S = LpVariable.dicts("S", range(months), lowBound=0, cat=LpContinuous) # Subcontracting

    # Coût unitaire de l'overtime
    ot_unit_cost = overtime_cost / p

    # Fonction objectif : Minimiser le coût total complet
    model += lpSum(
        W[t] * wage_r +
        H[t] * hiring +
        L[t] * layoff +
        I[t] * hold +
        O[t] * ot_unit_cost +
        S[t] * subcontracting_cost
        for t in range(months)
    ), "Total_Cost"

    # Contraintes
    for t in range(months):

        # 1. Contrainte de capacité de production régulière
        model += P[t] == W[t] * p, f"Reg_Prod_Capacity_Month_{t}"

        # 2. Limites d'heures supplémentaires
        if (t + 1) in overtime_months:
            model += O[t] <= W[t] * p * max_overtime, f"Max_Overtime_Month_{t}"
        else:
            model += O[t] == 0, f"No_Overtime_Month_{t}"

        # 3. Limites de sous-traitance
        if (t + 1) in subcontracting_months:
            model += S[t] <= max_contracting, f"Max_Subcontracting_Month_{t}"
        else:
            model += S[t] == 0, f"No_Subcontracting_Month_{t}"

        # 4. Équilibre de la main-d'œuvre et de l'inventaire (P + O + S)
        if t == 0:
            model += W[t] == w_i + H[t] - L[t], "Workforce_Balance_Month_0"
            model += I[t] == inv_i + P[t] + O[t] + S[t] - demand[t], "Inventory_Balance_Month_0"
        else:
            model += W[t] == W[t-1] + H[t] - L[t], f"Workforce_Balance_Month_{t}"
            model += I[t] == I[t-1] + P[t] + O[t] + S[t] - demand[t], f"Inventory_Balance_Month_{t}"

    # 5. Cible d'inventaire à la fin de l'horizon
    model += I[months - 1] >= inv_t, "Final_Inventory_Target"

    # Résolution
    model.solve()

    status = LpStatus[model.status]

    # Extraction des résultats
    if status == "Optimal":
        df = pd.DataFrame({
            "Month": np.arange(1, months+1),
            "Workers": [W[t].varValue for t in range(months)],
            "Hired": [H[t].varValue for t in range(months)],
            "Laid off": [L[t].varValue for t in range(months)],
            "Reg. Prod": [P[t].varValue for t in range(months)],
            "Overtime Prod": [O[t].varValue for t in range(months)],
            "Subcontracted": [S[t].varValue for t in range(months)],
            "Total Prod": [P[t].varValue + O[t].varValue + S[t].varValue for t in range(months)],
            "Demand": demand,
            "End Inventory": [I[t].varValue for t in range(months)],
        })

        summary = {
            "Total Wage": sum(W[t].varValue * wage_r for t in range(months)),
            "Total Hiring": sum(H[t].varValue * hiring for t in range(months)),
            "Total Layoff": sum(L[t].varValue * layoff for t in range(months)),
            "Total Overtime": sum(O[t].varValue * ot_unit_cost for t in range(months)),
            "Total Subcontracting": sum(S[t].varValue * subcontracting_cost for t in range(months)),
            "Total Inventory": sum(I[t].varValue * hold for t in range(months)),
            "Grand Total": value(model.objective)
        }
    else:
        df = pd.DataFrame()
        summary = {}

    return df, summary, status

In [ ]:
df_model4, summary_model4, status_model4 = solve_lp_model_4(demand, p, inv_i, inv_t, w_i, wage_r, hiring, layoff, hold, overtime_cost, max_overtime, overtime_months, subcontracting_cost, max_contracting, subcontracting_months)
display(df_model4)
display(summary_model4)

,Month,Workers,Hired,Laid off,Reg. Prod,Overtime Prod,Subcontracted,Total Prod,Demand,End Inventory
0,1,80.0,0.0,10.0,800.0,0.0,0.0,800.0,900,800.0
1,2,80.0,0.0,0.0,800.0,0.0,0.0,800.0,950,650.0
2,3,80.0,0.0,0.0,800.0,0.0,0.0,800.0,1200,250.0
3,4,80.0,0.0,0.0,800.0,0.0,0.0,800.0,1050,0.0
4,5,115.0,35.0,0.0,1150.0,0.0,0.0,1150.0,1100,50.0
5,6,125.0,10.0,0.0,1250.0,0.0,0.0,1250.0,1300,0.0
6,7,125.0,0.0,0.0,1250.0,0.0,0.0,1250.0,1250,0.0
7,8,125.0,0.0,0.0,1250.0,0.0,0.0,1250.0,1100,150.0
8,9,126.0,1.0,0.0,1260.0,4.0,0.0,1264.0,1300,114.0
9,10,173.0,47.0,0.0,1730.0,0.0,0.0,1730.0,1450,394.0


{'Total Wage': 10185000.0,
 'Total Hiring': 4650000.0,
 'Total Layoff': 250000.0,
 'Total Overtime': 490000.0,
 'Total Subcontracting': 0.0,
 'Total Inventory': 4032000.0,
 'Grand Total': 19607000.0}